<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller: Segmentar Motores Industriales sin Etiquetas 🏭🧩
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 04
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/04_Clustering_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 04 (Clustering No Supervisado)** con un problema nuevo: una planta industrial quiere descubrir, **sin etiquetas**, los *regímenes de operación* de sus motores y detectar trayectorias anómalas de sus vehículos internos. Al terminar serás capaz de:

1. **Preparar** datos para agrupar (escala y distancia) y **elegir $k$** en *K-Means* con la silueta, verificando la inercia con tu propio cálculo.
2. **Construir e interpretar** un agrupamiento **jerárquico** (enlaces, correlación cofenética, corte del dendrograma y anidación de las particiones).
3. **Usar DBSCAN** en datos no convexos con ruido, eligiendo $\varepsilon$ con las $k$-distancias, y **reconocer la trampa** de la silueta frente a la forma de los grupos.
4. **Medir la calidad y la estabilidad** de un agrupamiento: implementar el **ARI** desde la tabla de contingencia y evaluar la estabilidad entre semillas.

> 📍 **Antes de empezar:** repasa [*K-Means*](../04%20-%20Clustering%20No%20Supervisado/01_KMeans.ipynb), el [*Clustering Jerárquico*](../04%20-%20Clustering%20No%20Supervisado/02_Clustering_Jerarquico.ipynb), [*DBSCAN*](../04%20-%20Clustering%20No%20Supervisado/03_DBSCAN.ipynb) y las [*Métricas de Clustering*](../04%20-%20Clustering%20No%20Supervisado/04_Metricas_de_Clustering.ipynb). Si necesitas más guía, existe la [versión Para Dummies](Para%20Dummies/04_Clustering_Hands_On_Dummies.ipynb) de este taller.

---
## Instrucciones

1. **Resuelve de forma autónoma:** completa cada celda de código marcada con `# TODO:` respetando los **nombres de variables** que se piden (las autoverificaciones los usan).
2. Después de cada reto hay una celda de **autoverificación** con `assert`. Mientras el reto no esté resuelto **fallará con un mensaje** que te orienta; cuando todo esté bien imprimirá un ✅.
3. No borres ni modifiques las celdas de autoverificación ni la celda de preparación.
4. Antes de entregar ejecuta **Kernel → Restart & Run All** y comprueba que no queda ningún error ni ningún `# TODO` sin resolver.
5. Todas las respuestas escritas (conclusiones) van en **español** y deben apoyarse en los números que obtuviste.

---
## Preparación del entorno y de los datos ⚙️

Ejecuta la celda siguiente (**no la modifiques**). Genera, con semilla fija y sin descargar nada:

- `motores` / `X` / `y_regimen`: 600 motores con 4 variables (`vibracion`, `temperatura`, `horas_uso`, `corriente`) que pertenecen a **3 regímenes de operación**. `horas_uso` está en miles de horas y **no informa** el régimen. `y_regimen` es la verdad de referencia (en la vida real no existiría; aquí solo sirve para evaluar).
- `idx_sub`: índices de una submuestra de 120 motores para el análisis jerárquico (Reto 2).
- `Xa` / `ym` / `es_normal`: trayectorias 2D con forma de dos *medias lunas* más 15 puntos anómalos (Reto 3).

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, fcluster, cophenet
from scipy.spatial.distance import pdist

SEED = 2024
rng = np.random.default_rng(SEED)


def listo(*valores):
    """True si ninguna variable sigue sin resolver (None o ...)."""
    return all(v is not None and v is not Ellipsis for v in valores)


# --- Retos 1, 2 y 4: 600 motores en 3 regimenes ---------------------------------
Xb, y_regimen = make_blobs(n_samples=600, centers=[[0, 0], [4, 0], [2, 3.5]], cluster_std=0.8, random_state=SEED)
motores = pd.DataFrame({
    "vibracion": Xb[:, 0],
    "temperatura": Xb[:, 1],
    "horas_uso": rng.normal(8000, 1000, 600),          # escala enorme y SIN informacion del regimen
    "corriente": Xb[:, 0] * 0.5 + rng.normal(0, 0.5, 600),
})
X = motores.to_numpy()
idx_sub = rng.choice(600, 120, replace=False)

# --- Reto 3: trayectorias en forma de medias lunas + 15 anomalias ----------------
Xm, ym = make_moons(500, noise=0.06, random_state=SEED)
anomalias = np.column_stack([rng.uniform(-1.5, 2.5, 15), rng.uniform(-1.2, 1.7, 15)])
Xa = np.vstack([Xm, anomalias])
es_normal = np.r_[np.ones(500, dtype=bool), np.zeros(15, dtype=bool)]

print("Motores:", X.shape, "| regimenes:", np.bincount(y_regimen), "| trayectorias:", Xa.shape)
display(motores.describe().round(1).loc[["mean", "std"]])

---
## Reto 1: Escala, elección de $k$ e inercia en *K-Means* 🎯 (20 puntos)

**Contexto.** El jefe de mantenimiento quiere agrupar los 600 motores por su comportamiento, pero `horas_uso` (miles de horas) convive con variables de escala unitaria. Debes demostrar con datos cuánto importa la escala y justificar el número de grupos.

**Enunciado**

1. Ajusta `KMeans(n_clusters=3, n_init=10, random_state=SEED)` sobre `X` **sin escalar** y calcula `ari_sin` (ARI de sus etiquetas frente a `y_regimen`).
2. Estandariza con `StandardScaler` y guarda la matriz en `Xs`. Repite el ajuste y calcula `ari_con`.
3. Con `Xs`, calcula la **silueta media** para $k=2,\dots,7$ (mismo `KMeans`, `n_init=10`, `random_state=SEED`) en el diccionario `silueta_k` y guarda en `k_mejor` el $k$ con mayor silueta.
4. Escribe `inercia_manual(X, etiquetas, centros)`, que calcule **a mano** (con NumPy) la inercia (suma de distancias euclídeas al cuadrado de cada punto a su centroide), y compárala con el atributo `inertia_` del modelo `modelo_final` (`KMeans(k_mejor)` ajustado sobre `Xs`).

**Criterios de aceptación**

- `ari_sin` es casi nulo y `ari_con` es alto: la escala cambia por completo el resultado.
- `silueta_k` tiene las 6 claves y `k_mejor` es coherente con ella.
- `inercia_manual` coincide con `modelo_final.inertia_` (error relativo $<10^{-6}$).

In [ ]:
# TODO 1: K-Means (k=3) sobre X SIN escalar y su ARI frente a y_regimen
ari_sin = None

# TODO 2: estandariza X -> Xs y repite el ajuste; guarda el ARI en ari_con
Xs = None
ari_con = None

# TODO 3: silueta media para k = 2..7 sobre Xs
silueta_k = {}          # {k: silueta media}
k_mejor = None          # el k con mayor silueta


# TODO 4: inercia calculada a mano
def inercia_manual(X, etiquetas, centros):
    """Suma de ||x_i - mu_{c(i)}||^2 sobre todos los puntos."""
    pass


modelo_final = None     # KMeans(n_clusters=k_mejor, n_init=10, random_state=SEED) ajustado sobre Xs

In [ ]:
# ---- Autoverificacion del Reto 1 (no modificar) ----
assert listo(ari_sin, ari_con), "Reto 1: calcula ari_sin y ari_con (pasos 1 y 2)."
assert ari_sin < 0.2, f"Reto 1: sin escalar el ARI deberia ser casi nulo (< 0.2) y obtuviste {ari_sin:.3f}. Revisa que uses X, no Xs."
assert listo(Xs) and np.asarray(Xs).shape == X.shape, "Reto 1: Xs debe tener la misma forma que X (estandariza las 4 columnas)."
assert np.allclose(np.asarray(Xs).mean(axis=0), 0, atol=1e-6) and np.allclose(np.asarray(Xs).std(axis=0), 1, atol=1e-6), "Reto 1: Xs no esta estandarizada (media 0, desviacion 1)."
assert ari_con > 0.85, f"Reto 1: con datos estandarizados el ARI deberia superar 0.85 y obtuviste {ari_con:.3f}."
assert sorted(silueta_k) == [2, 3, 4, 5, 6, 7], "Reto 1: silueta_k debe tener como claves k = 2, 3, ..., 7."
assert all(-1 <= v <= 1 for v in silueta_k.values()), "Reto 1: una silueta media siempre esta en [-1, 1]."
assert k_mejor == max(silueta_k, key=silueta_k.get), "Reto 1: k_mejor debe ser el k de mayor silueta en silueta_k."
assert k_mejor == 3, f"Reto 1: con 3 regimenes reales la silueta deberia preferir k = 3 y obtuviste k = {k_mejor}. Revisa que uses Xs."
assert listo(modelo_final) and modelo_final.n_clusters == k_mejor, "Reto 1: modelo_final debe ser un KMeans con n_clusters = k_mejor, ya ajustado."
prueba = inercia_manual(Xs, modelo_final.labels_, modelo_final.cluster_centers_)
assert prueba is not None and abs(prueba - modelo_final.inertia_) / modelo_final.inertia_ < 1e-6, "Reto 1: inercia_manual no coincide con modelo_final.inertia_. Recuerda elevar al cuadrado y sumar sobre TODAS las columnas."
print(f"✅ Reto 1 correcto | ARI sin escalar = {ari_sin:.3f} | ARI estandarizado = {ari_con:.3f} | k elegido = {k_mejor}")

---
## Reto 2: Jerarquía, enlaces y anidación 🌳 (20 puntos)

**Contexto.** Para auditar a mano un grupo pequeño de motores, el equipo prefiere un **árbol** (dendrograma) en lugar de una partición plana. Trabajarás con la submuestra de 120 motores ya estandarizados.

**Enunciado**

1. Define `Xsub = Xs[idx_sub]` y `ysub = y_regimen[idx_sub]` (usa el `Xs` del Reto 1).
2. Para cada enlace de `metodos` (`single`, `complete`, `average`, `ward`) construye la matriz de enlace con `linkage`, calcula la **correlación cofenética** y guárdala en el diccionario `cofenetica`; corta el árbol en 3 grupos con `fcluster(..., criterion="maxclust")` y guarda el ARI frente a `ysub` en `ari_cortes`.
3. Guarda en `Z_ward` la matriz de enlace de Ward y en `mejor_enlace` el método con mayor ARI.
4. Comprueba la **anidación**: con `Z_ward`, obtén `lab3` y `lab4` (cortes en 3 y 4 grupos) y guarda en `tabla_cruce` la tabla `pd.crosstab(lab3, lab4)`.

**Criterios de aceptación**

- Los valores de `cofenetica` y `ari_cortes` coinciden con un cálculo independiente.
- Ward recupera bien los regímenes (ARI $>0{,}7$) y el enlace *single* sufre el encadenamiento (su ARI es menor que el de Ward).
- En `tabla_cruce` **cada grupo de 4 cae dentro de un solo grupo de 3** (la jerarquía es anidada).

In [ ]:
Xsub = None      # TODO: la submuestra estandarizada
ysub = None      # TODO: las etiquetas de referencia de la submuestra

metodos = ["single", "complete", "average", "ward"]
cofenetica = {}  # TODO: {metodo: correlacion cofenetica}  (pista: cophenet(Z, pdist(Xsub)))
ari_cortes = {}  # TODO: {metodo: ARI del corte en 3 grupos frente a ysub}
# TODO: recorre `metodos` y llena ambos diccionarios

Z_ward = None        # TODO: linkage(Xsub, "ward")
mejor_enlace = None  # TODO: el metodo con mayor ARI en ari_cortes
tabla_cruce = None   # TODO: pd.crosstab(lab3, lab4) con los cortes de Z_ward en 3 y 4 grupos

In [ ]:
# ---- Autoverificacion del Reto 2 (no modificar) ----
assert listo(Xsub, ysub, Z_ward, mejor_enlace, tabla_cruce), "Reto 2: faltan variables por definir (Xsub, ysub, Z_ward, mejor_enlace, tabla_cruce)."
_Xref = StandardScaler().fit_transform(X)[idx_sub]
assert np.allclose(Xsub, _Xref), "Reto 2: Xsub debe ser Xs[idx_sub] con Xs estandarizada (Reto 1)."
assert sorted(cofenetica) == sorted(metodos) and sorted(ari_cortes) == sorted(metodos), "Reto 2: cofenetica y ari_cortes deben tener como claves los 4 enlaces."
for m in metodos:
    _Z = linkage(_Xref, m)
    _c = cophenet(_Z, pdist(_Xref))[0]
    _a = adjusted_rand_score(y_regimen[idx_sub], fcluster(_Z, 3, criterion="maxclust"))
    assert abs(cofenetica[m] - _c) < 1e-6, f"Reto 2: la correlacion cofenetica de '{m}' no coincide (esperada {_c:.3f}, tuya {cofenetica[m]:.3f})."
    assert abs(ari_cortes[m] - _a) < 1e-6, f"Reto 2: el ARI del corte en 3 grupos de '{m}' no coincide (esperado {_a:.3f}, tuyo {ari_cortes[m]:.3f}). Usa criterion='maxclust'."
assert np.asarray(Z_ward).shape == (len(idx_sub) - 1, 4), "Reto 2: Z_ward debe tener forma (n-1, 4)."
assert ari_cortes["ward"] > 0.7, f"Reto 2: Ward deberia superar ARI 0.7 y obtuviste {ari_cortes['ward']:.3f}."
assert ari_cortes["single"] < ari_cortes["ward"], "Reto 2: el enlace single (encadenamiento) deberia quedar por debajo de Ward."
assert mejor_enlace == max(ari_cortes, key=ari_cortes.get), "Reto 2: mejor_enlace debe ser la clave de mayor valor en ari_cortes."
_t = np.asarray(tabla_cruce)
assert _t.shape == (3, 4), f"Reto 2: tabla_cruce debe ser de 3 filas (corte en 3) x 4 columnas (corte en 4); la tuya es {_t.shape}."
assert ((_t > 0).sum(axis=0) == 1).all(), "Reto 2: en una jerarquia anidada cada grupo del corte en 4 cae en UN solo grupo del corte en 3."
print(f"✅ Reto 2 correcto | mejor enlace por ARI: {mejor_enlace} | cofenetica Ward = {cofenetica['ward']:.3f}")

---
## Reto 3: DBSCAN, ruido y la trampa de la silueta 🌙 (25 puntos)

**Contexto.** Los vehículos internos de la planta describen trayectorias con forma de **dos medias lunas**, y a veces el GPS registra puntos anómalos. *K-Means* corta las lunas en dos mitades equivocadas; *DBSCAN* debería seguirlas y marcar las anomalías como ruido.

**Enunciado**

1. Calcula la distancia de cada punto de `Xa` a su **5.º vecino más cercano** (sin contar el propio punto) y guarda las distancias ordenadas en `kdist`. Fija `eps` como el **percentil 95** de `kdist`.
2. Ajusta `DBSCAN(eps=eps, min_samples=5)` y guarda las etiquetas en `etq_db`; calcula `n_grupos` (sin contar el ruido) y `n_ruido` (puntos con etiqueta $-1$).
3. Ajusta `KMeans(2, n_init=10, random_state=SEED)` y guarda sus etiquetas en `etq_km`.
4. Calcula, **solo sobre los puntos normales** (`es_normal`), `ari_db` y `ari_km` frente a `ym`.
5. Calcula `sil_km` (silueta de `etq_km` sobre los puntos normales) y `sil_db` (silueta de `etq_db` **solo sobre los puntos que DBSCAN no marcó como ruido**).

**Criterios de aceptación**

- `eps` razonable ($0{,}05<\varepsilon<0{,}4$), exactamente 2 grupos y una cantidad plausible de ruido.
- DBSCAN recupera las lunas ($\text{ARI}>0{,}9$) y *K-Means* no ($\text{ARI}<0{,}6$).
- **La trampa:** a pesar de ello, `sil_km > sil_db`. Explícala en las conclusiones.

In [ ]:
# TODO 1: distancia al 5.o vecino (sin contar el propio punto), ordenada, y eps = percentil 95
kdist = None
eps = None

# TODO 2: DBSCAN(eps=eps, min_samples=5)
etq_db = None
n_grupos = None
n_ruido = None

# TODO 3: K-Means con 2 grupos
etq_km = None

# TODO 4: ARI sobre los puntos normales (es_normal)
ari_db = None
ari_km = None

# TODO 5: siluetas (sil_db SIN los puntos de ruido de DBSCAN)
sil_km = None
sil_db = None

In [ ]:
# ---- Autoverificacion del Reto 3 (no modificar) ----
assert listo(kdist, eps, etq_db, n_grupos, n_ruido, etq_km, ari_db, ari_km, sil_km, sil_db), "Reto 3: faltan variables por definir (revisa los 5 pasos)."
assert len(kdist) == len(Xa) and np.all(np.diff(kdist) >= 0), "Reto 3: kdist debe tener una distancia por punto y estar ordenada de menor a mayor."
assert abs(eps - np.percentile(kdist, 95)) < 1e-9, "Reto 3: eps debe ser el percentil 95 de kdist."
assert 0.05 < eps < 0.4, f"Reto 3: eps = {eps:.3f} esta fuera del rango razonable (0.05, 0.4). Revisa que la k-distancia use el 5.o vecino SIN contar el propio punto."
assert len(etq_db) == len(Xa) and len(etq_km) == len(Xa), "Reto 3: las etiquetas deben tener una entrada por punto de Xa (incluidas las anomalias)."
assert n_grupos == 2, f"Reto 3: DBSCAN deberia hallar 2 grupos (las lunas) y encontraste {n_grupos}."
assert n_ruido == int((np.asarray(etq_db) == -1).sum()), "Reto 3: n_ruido debe contar las etiquetas -1 de etq_db."
assert 3 <= n_ruido <= 40, f"Reto 3: se esperaban entre 3 y 40 puntos de ruido (hay 15 anomalias) y obtuviste {n_ruido}."
assert (np.asarray(etq_db)[~es_normal] == -1).sum() >= 7, "Reto 3: DBSCAN deberia marcar como ruido al menos 7 de las 15 anomalias."
assert ari_db > 0.9, f"Reto 3: el ARI de DBSCAN sobre los puntos normales deberia superar 0.9 y fue {ari_db:.3f}."
assert ari_km < 0.6, f"Reto 3: el ARI de K-Means en las lunas deberia ser bajo (< 0.6) y fue {ari_km:.3f}. Recuerda evaluar solo sobre es_normal."
_ok = np.asarray(etq_db) != -1
assert abs(sil_db - silhouette_score(Xa[_ok], np.asarray(etq_db)[_ok])) < 1e-6, "Reto 3: sil_db debe calcularse solo con los puntos que NO son ruido (etiqueta distinta de -1)."
assert abs(sil_km - silhouette_score(Xa[es_normal], np.asarray(etq_km)[es_normal])) < 1e-6, "Reto 3: sil_km debe calcularse solo sobre los puntos normales."
assert sil_km > sil_db, "Reto 3: la trampa de la silueta: K-Means (partición equivocada) deberia obtener MAYOR silueta que DBSCAN (particion correcta)."
print(f"✅ Reto 3 correcto | ARI DBSCAN = {ari_db:.3f} vs K-Means = {ari_km:.3f} | silueta K-Means = {sil_km:.3f} > DBSCAN = {sil_db:.3f}")

---
## Reto 4: ARI desde cero y estabilidad entre semillas ⚖️ (20 puntos)

**Contexto.** Sin etiquetas de verdad, una forma de desconfiar de un agrupamiento es ver si **cambia al cambiar la semilla**. Para compararlo necesitas el **índice de Rand ajustado (ARI)**, que implementarás tú mismo a partir de la **tabla de contingencia** $n_{ij}$ (puntos que están a la vez en el grupo $i$ de la partición $a$ y en el grupo $j$ de la partición $b$):

$$
\mathrm{ARI}=\frac{\sum_{ij}\binom{n_{ij}}{2}-\dfrac{\sum_i\binom{a_i}{2}\sum_j\binom{b_j}{2}}{\binom{n}{2}}}{\dfrac12\left[\sum_i\binom{a_i}{2}+\sum_j\binom{b_j}{2}\right]-\dfrac{\sum_i\binom{a_i}{2}\sum_j\binom{b_j}{2}}{\binom{n}{2}}}
$$

donde $a_i$ y $b_j$ son las sumas de filas y de columnas de la tabla.

**Enunciado**

1. Implementa `ari_propio(a, b)` usando `pd.crosstab` (o `np.unique` + `np.add.at`) y `math.comb`/`n*(n-1)/2`.
2. Para $k=2,\dots,6$ entrena 6 corridas de `KMeans(k, n_init=1, random_state=s)` con $s=0,\dots,5$ sobre `Xs`, calcula el ARI (con `ari_propio`) entre **todos los pares** de corridas (15 pares) y guarda su promedio en `estab[k]`.

**Criterios de aceptación**

- `ari_propio` coincide con `adjusted_rand_score` en tres pares de particiones (incluida una permutación de etiquetas, que debe dar 1, y dos particiones al azar, que deben dar casi 0).
- `estab` tiene las claves 2 a 6, valores en $[-1,1]$, y para $k=3$ (el número real de regímenes) la estabilidad es alta ($\ge 0{,}7$) y coincide con el cálculo de referencia.

In [ ]:
def ari_propio(a, b):
    """ARI calculado desde la tabla de contingencia, sin usar sklearn.metrics."""
    # TODO: 1) tabla de contingencia  2) sumas de combinaciones C(x, 2) por celda, fila y columna
    #       3) aplica la formula del enunciado
    pass


estab = {}   # TODO: {k: ARI medio entre los 15 pares de corridas}
# Pista: etiquetas = [KMeans(k, n_init=1, random_state=s).fit_predict(Xs) for s in range(6)]
#        y los pares con itertools.combinations(etiquetas, 2)

In [ ]:
# ---- Autoverificacion del Reto 4 (no modificar) ----
_rg = np.random.default_rng(0)
_Xr = StandardScaler().fit_transform(X)
_et = KMeans(3, n_init=10, random_state=SEED).fit_predict(_Xr)
_pares = [(y_regimen, _et), (y_regimen, (y_regimen + 1) % 3), (_rg.integers(0, 4, 300), _rg.integers(0, 4, 300))]
for _i, (_a, _b) in enumerate(_pares):
    _v = ari_propio(_a, _b)
    assert _v is not None, "Reto 4: ari_propio todavia no devuelve ningun valor."
    assert abs(_v - adjusted_rand_score(_a, _b)) < 1e-9, f"Reto 4: ari_propio no coincide con sklearn en el par {_i + 1} (el tuyo {_v:.4f}, sklearn {adjusted_rand_score(_a, _b):.4f})."
assert abs(ari_propio(y_regimen, (y_regimen + 1) % 3) - 1) < 1e-9, "Reto 4: renombrar los grupos no debe cambiar el ARI (debe dar 1)."
assert sorted(estab) == [2, 3, 4, 5, 6], "Reto 4: estab debe tener como claves k = 2, 3, 4, 5, 6."
assert all(-1 <= v <= 1 for v in estab.values()), "Reto 4: un ARI siempre esta en [-1, 1]."
assert estab[3] >= 0.7, f"Reto 4: para k = 3 la estabilidad deberia ser >= 0.7 y obtuviste {estab[3]:.3f}."
for _k in (3, 5):
    _et = [KMeans(_k, n_init=1, random_state=s).fit_predict(_Xr) for s in range(6)]
    _ref = np.mean([adjusted_rand_score(p, q) for p, q in itertools.combinations(_et, 2)])
    assert abs(estab[_k] - _ref) < 1e-6, f"Reto 4: estab[{_k}] = {estab[_k]:.4f} no coincide con la referencia {_ref:.4f}. Usa 6 corridas (semillas 0-5), n_init=1 y los 15 pares, sobre Xs."
print("✅ Reto 4 correcto | estabilidad por k:", {k: round(v, 3) for k, v in estab.items()})

---
## Reto 5: Conclusiones 🧠 (15 puntos)

*Sin código.* Escribe entre **5 y 8 líneas** que interpreten **tus** resultados (cita cifras de tus variables `ari_sin`, `ari_con`, `silueta_k`, `cofenetica`, `ari_db`, `sil_km`, `sil_db`, `estab`). Debes responder, como mínimo:

1. ¿Por qué cambió tanto el ARI de *K-Means* al estandarizar y qué habría pasado en la planta si se hubiera agrupado con las unidades originales?
2. ¿Qué enlace elegirías para el árbol de los motores y por qué el *single* falla aquí?
3. ¿Por qué la silueta prefirió la partición equivocada de las lunas? ¿Qué métrica o criterio usarías en su lugar cuando los grupos no son convexos?
4. ¿Qué te dice la estabilidad sobre el número de grupos? ¿Es una prueba de que $k=3$ es "la verdad"?

**Tus conclusiones (edita esta celda):**

> *(escribe aquí)*

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: escala, selección de $k$ por silueta e inercia propia | 20 |
| Reto 2: enlaces, correlación cofenética, corte y anidación del dendrograma | 20 |
| Reto 3: DBSCAN con $\varepsilon$ por $k$-distancias, ruido y trampa de la silueta | 25 |
| Reto 4: ARI desde la tabla de contingencia y estabilidad entre semillas | 20 |
| Reto 5: conclusiones (5-8 líneas, con cifras propias y razonamiento correcto) | 15 |
| **Total** | **100** |

*Las autoverificaciones comprueban propiedades y rangos; el docente puede revisar además la calidad del código y de la interpretación.*

### ✅ Checklist de entrega

- [ ] Ejecuté **Restart & Run All** y no hay errores ni celdas `# TODO` sin resolver.
- [ ] Las cuatro autoverificaciones imprimen ✅.
- [ ] Respondí las cuatro preguntas de las conclusiones con cifras de mis propios resultados.
- [ ] No modifiqué las celdas de preparación ni de autoverificación.
- [ ] Entregué el notebook con los resultados visibles (salidas guardadas).

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>